In [1]:
from neo4j import GraphDatabase
import pandas as pd
import psycopg

In [2]:
# This code was made by Jake S. to load the playlist graph data
class SpotifyGraphSetup:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def create_schema(self):
        """Cleans up old constraints by name and creates new ones using exact property names."""
        with self.driver.session() as session:
            result = session.run("SHOW CONSTRAINTS YIELD name")
            constraint_names = [record["name"] for record in result]
            
            if constraint_names:
                print("Cleaning up old constraints...")
                for name in constraint_names:
                    session.run(f"DROP CONSTRAINT {name}")
                    print(f" - Dropped: {name}")

            queries = [
                "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Song) REQUIRE s.song_uri IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Artist) REQUIRE a.artist_uri IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Playlist) REQUIRE p.playlist_uri IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Type) REQUIRE t.name IS UNIQUE"
            ]
            
            for query in queries:
                session.run(query)
                
        print("Schema constraints created successfully.")

    def load_data(self):
        """Reads the CSV files from the import folder and builds the graph using batched transactions."""
        
        node_queries = [
            """LOAD CSV WITH HEADERS FROM 'file:///song.csv' AS row
               CALL {
                   WITH row
                   MERGE (s:Song {song_uri: row.song_uri})
                   SET s.title = row.name, 
                       s.popularity = toInteger(row.popularity), 
                       s.release_date = row.release_date
               } IN TRANSACTIONS OF 10000 ROWS""", 
               
            """LOAD CSV WITH HEADERS FROM 'file:///artist.csv' AS row
               CALL {
                   WITH row
                   MERGE (a:Artist {artist_uri: row.artist_uri})
                   SET a.name = row.artist_name, 
                       a.followers = toInteger(row.artist_followers), 
                       a.popularity = toInteger(row.artist_popularity)
               } IN TRANSACTIONS OF 10000 ROWS""",
               
            """LOAD CSV WITH HEADERS FROM 'file:///playlist.csv' AS row
               CALL {
                   WITH row
                   MERGE (p:Playlist {playlist_uri: row.playlist_uri})
                   SET p.name = row.name, 
                       p.followers = toInteger(row.playlist_followers), 
                       p.n_tracks = toInteger(row.n_tracks)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///user.csv' AS row
               CALL {
                   WITH row
                   MERGE (u:User {user_id: row.user})
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///genre.csv' AS row
               CALL {
                   WITH row
                   MERGE (g:Genre {name: row.artist_genres})
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///type.csv' AS row
               CALL {
                   WITH row
                   MERGE (t:Type {name: row.type})
               } IN TRANSACTIONS OF 10000 ROWS"""
        ]

        relationship_queries = [
            """LOAD CSV WITH HEADERS FROM 'file:///sing.csv' AS row
               CALL {
                   WITH row
                   MATCH (a:Artist {artist_uri: row.artists_uris}), (s:Song {song_uri: row.song_uri})
                   MERGE (a)-[:SING]->(s)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///created.csv' AS row
               CALL {
                   WITH row
                   MATCH (u:User {user_id: row.user}), (p:Playlist {playlist_uri: row.playlist_uri})
                   MERGE (u)-[:CREATED]->(p)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///in.csv' AS row
               CALL {
                   WITH row
                   MATCH (s:Song {song_uri: row.song_uri}), (p:Playlist {playlist_uri: row.playlist_uris})
                   MERGE (s)-[:IN]->(p)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///of_type.csv' AS row
               CALL {
                   WITH row
                   MATCH (p:Playlist {playlist_uri: row.playlist_uri}), (t:Type {name: row.type})
                   MERGE (p)-[:OFTYPE]->(t)
               } IN TRANSACTIONS OF 10000 ROWS""",

            """LOAD CSV WITH HEADERS FROM 'file:///labelled.csv' AS row
               CALL {
                   WITH row
                   MATCH (a:Artist {artist_uri: row.artist_uri}), (g:Genre {name: row.artist_genres})
                   MERGE (a)-[:LABELLED]->(g)
               } IN TRANSACTIONS OF 10000 ROWS"""
        ]

        with self.driver.session() as session:
            print("Loading nodes in batches...")
            for query in node_queries:
                session.run(query)
            
            print("Loading relationships in batches (this might take a moment)...")
            for query in relationship_queries:
                session.run(query)
                
        print("Graph data loaded successfully!")

In [3]:
neo4j_uri = "bolt://localhost:7687"
neo4j_user = "neo4j"
neo4j_password = "password"

setup = SpotifyGraphSetup(neo4j_uri, neo4j_user, neo4j_password)
setup.create_schema()
setup.load_data() 
setup.close()

Cleaning up old constraints...
 - Dropped: constraint_5ac005f2
 - Dropped: constraint_60d3be4d
 - Dropped: constraint_bb4804ff
 - Dropped: constraint_d7f52273
 - Dropped: constraint_f4d47ccd
 - Dropped: constraint_f6ec32a1
Schema constraints created successfully.
Loading nodes in batches...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=2, column=16, offset=68>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 68, 'line': 2, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "LOAD CSV WITH HEADERS FROM 'file:///song.csv' AS row\n               CALL {\n                   WITH row\n                   MERGE (s:Song {song_uri: row.song_uri})\n                   SET s.title = row.name, \n                       s.popularity = toInteger(row.popularity), \n                       s.release_date = row.release_date\n    

Loading relationships in batches (this might take a moment)...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=2, column=16, offset=68>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 68, 'line': 2, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "LOAD CSV WITH HEADERS FROM 'file:///sing.csv' AS row\n               CALL {\n                   WITH row\n                   MATCH (a:Artist {artist_uri: row.artists_uris}), (s:Song {song_uri: row.song_uri})\n                   MERGE (a)-[:SING]->(s)\n               } IN TRANSACTIONS OF 10000 ROWS"
Received notification from DBMS server: 

Graph data loaded successfully!


In [16]:
conninfo = "host=localhost port=5432 dbname=202HW3 user=postgres password="

In [5]:
def execute_sql(sql, params=None):
    """
    For creating tables, inserting into tables.
    """
    with psycopg.connect(conninfo) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
        conn.commit()

def query_df(sql, params=None):
    """
    For running queries. Returns pandas dataframe of query ouput.
    """
    with psycopg.connect(conninfo) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [d.name for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

In [6]:
create_table_sql = """
    DROP TABLE IF EXISTS spotify_charts;
    CREATE TABLE spotify_charts (
        id SERIAL PRIMARY KEY,
        title TEXT,
        rank INT,
        chart_date DATE,
        artist TEXT,
        url TEXT,
        region VARCHAR(100),
        chart VARCHAR(50),
        trend VARCHAR(50),
        streams NUMERIC
    );
    """
execute_sql(create_table_sql)

In [7]:
def load_csv(file_path, table_name):
    with psycopg.connect(conninfo) as conn:
        with conn.cursor() as cur:
            with open(file_path, 'r', encoding='utf-8') as f:
                copy_command = f"COPY {table_name} (title, rank, chart_date, artist, url, region, chart, trend, streams) FROM STDIN WITH (FORMAT CSV, HEADER TRUE)"
                with cur.copy(copy_command) as copy:
                    while data := f.read(8192):
                        copy.write(data)
            conn.commit()

load_csv('../data/charts.csv', 'spotify_charts')
print('Done')

Done


In [8]:
def genre_marketing_report(artist_name):
    """
    This function will output a plan for an artist according to their trend on the top200 chart.
    If not on the chart will print a statement to explain so.
    If the artist has fallen off the chart it recommends artists to collaborate with and what 
    new genres they tap into that the input artist does not.
    If the artist has more upward trend labels than downward trend labels over all their charted songs,
    it will tell them to use our tool from task 1: regional analysis tool.
    If the artist has more downward trend labels than upward over all their charted songs,
    it will advise them to release a new song aimed at a new niche genre.
    """
    DATA_CUTOFF_DATE = pd.to_datetime('2021-12-31').date()

    # Relational Data Portion
    with psycopg.connect(conninfo) as conn:
        with conn.cursor() as cur: 
            status_sql = """
            SELECT DISTINCT ON (chart, region, title) chart, region, rank, trend, chart_date
            FROM spotify_charts
            WHERE artist ILIKE %s 
              AND chart IN ('top200', 'viral50')
            ORDER BY chart, region, title, chart_date DESC, rank ASC;
            """
            cur.execute(status_sql, (f'%{artist_name}%',))
            records = cur.fetchall()

    if not records:
        print(f"No chart data found for {artist_name}.")
        return

    up_count = 0
    down_count = 0
    current_song_count = 0
    global_last_seen = records[0][4]

    for r in records:
        chart, region, rank, trend, chart_date = r
        if chart_date > global_last_seen:
            global_last_seen = chart_date
            
        if chart_date >= DATA_CUTOFF_DATE:
            current_song_count += 1
            if trend in ['MOVE_UP', 'NEW_ENTRY']:
                up_count += 1
            elif trend in 'MOVE_DOWN':
                down_count += 1

    if global_last_seen < DATA_CUTOFF_DATE:
        plan = "Return to Chart"
    elif down_count >= up_count:
        plan = "Revert Stalling"
    else:
        plan = "Maintain Momentum"

    last_seen_str = str(global_last_seen) + (" (end of data)" if global_last_seen == DATA_CUTOFF_DATE else "")
    print(f"--- Strategic Analysis: {artist_name.upper()} ---")
    print(f"Active Top200 Chart Entries: {current_song_count}")
    print(f"Status: Last seen {last_seen_str} | Plan: {plan}")

    # Graph Data Portion
    driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
    with driver.session() as session:
        my_genres_res = session.run("MATCH (a:Artist) WHERE a.name CONTAINS $n MATCH (a)-[:LABELLED]->(g:Genre) RETURN collect(DISTINCT g.name)", n=artist_name).single()
        my_genres = my_genres_res[0] if my_genres_res and my_genres_res[0] else []
        
        bridge_query = """
        MATCH (a:Artist) WHERE a.name CONTAINS $artist
        MATCH (a)-[:SING]->(:Song)-[:IN]->(p:Playlist)<-[:IN]-(other:Song)<-[:SING]-(partner:Artist)
        WHERE NOT partner.name CONTAINS $artist
        MATCH (partner)-[:LABELLED]->(g:Genre)
        RETURN partner.name as Partner, collect(DISTINCT g.name) as PartnerGenres, count(DISTINCT p) as Shared
        ORDER BY Shared DESC
        """
        recs = session.run(bridge_query, artist=artist_name).data()
    driver.close()

    # Print outs
    final_partners = []
    for r in recs:
        new_genres = [g for g in r['PartnerGenres'] if g not in my_genres]
        if new_genres:
            final_partners.append({
                'name': r['Partner'],
                'shared': r['Shared'],
                'new': new_genres
            })
        if len(final_partners) == 5:
            break

    print("-" * 40)
    
    if plan == "Maintain Momentum":
        print(f"{artist_name.upper()} is currently dominating the Top 200 charts.")
        print("To optimize your current lead, please utilize the 'Region Analysis Tool'.")

    elif plan == "Revert Stalling":
        if final_partners:
            all_new_genres = set()
            for p in final_partners:
                all_new_genres.update(p['new'])
            
            print(f"ADVICE: {artist_name.upper()} is losing momentum. Consider releasing a new song to target these new genres:")
            print(f"{', '.join(all_new_genres)}")
        else:
            print("Momentum is stalling, but no clear genre bridges were found.")

    elif plan == "Return to Chart":
        if final_partners:
            print(f"RECOVERY PLAN: Proposed collabs to facilitate a return to the Top 200:")
            for p in final_partners:
                print(f"- {p['name']} ({p['shared']} shared playlists)")
                print(f"  New Genre(s): {', '.join(p['new'])}")
        else:
            print(f"No proposed partners found to help {artist_name.upper()} return to the charts.")

In [9]:
genre_marketing_report('Justin Bieber')

--- Strategic Analysis: JUSTIN BIEBER ---
Active Top200 Chart Entries: 7
Status: Last seen 2021-12-31 (end of data) | Plan: Maintain Momentum
----------------------------------------
JUSTIN BIEBER is currently dominating the Top 200 charts.
To optimize your current lead, please utilize the 'Region Analysis Tool'.


In [10]:
genre_marketing_report('Drake')

--- Strategic Analysis: DRAKE ---
Active Top200 Chart Entries: 18
Status: Last seen 2021-12-31 (end of data) | Plan: Revert Stalling
----------------------------------------
ADVICE: DRAKE is losing momentum. Consider releasing a new song to target these new genres:
southern hip hop, chicago rap, west coast rap, trap, slap house, conscious hip hop, atl hip hop


In [11]:
genre_marketing_report('Fall Out Boy')

--- Strategic Analysis: FALL OUT BOY ---
Active Top200 Chart Entries: 0
Status: Last seen 2019-12-05 | Plan: Return to Chart
----------------------------------------
RECOVERY PLAN: Proposed collabs to facilitate a return to the Top 200:
- Imagine Dragons (52 shared playlists)
  New Genre(s): rock
- blink-182 (48 shared playlists)
  New Genre(s): rock, pop punk, punk, socal pop punk
- Green Day (44 shared playlists)
  New Genre(s): permanent wave, punk
- Linkin Park (43 shared playlists)
  New Genre(s): alternative metal, nu metal, post-grunge, rap metal
- The Killers (39 shared playlists)
  New Genre(s): rock, alternative rock, permanent wave, dance rock


In [12]:
genre_marketing_report('Title Fight')

No chart data found for Title Fight.


In [13]:
def print_curr_chart_data(artist_name):
    """
    This function outputs the region, song name, rank, and trend of all charting songs
    for a input artist at the cutoff data of the data.
    It also counts the number of upward trend labels and downward trend labels.
    """
    DATA_CUTOFF_DATE = pd.to_datetime('2021-12-31').date()
    
    with psycopg.connect(conninfo) as conn:
        with conn.cursor() as cur:
            # FIXED: Added 'title' to DISTINCT ON and ORDER BY to get EVERY song
            # (Note: I put 'title' at the end of the SELECT list so it doesn't break your r[x] indexes below)
            status_sql = """
            SELECT DISTINCT ON (chart, region, title) chart, region, rank, trend, chart_date, title
            FROM spotify_charts
            WHERE artist ILIKE %s 
              AND chart IN ('top200', 'viral50')
            ORDER BY chart, region, title, chart_date DESC, rank ASC;
            """
            cur.execute(status_sql, (f'%{artist_name}%',))
            records = cur.fetchall()

    if not records:
        print(f"No chart data found for {artist_name}.")
        return

    current_records = [r for r in records if r[4] >= DATA_CUTOFF_DATE]

    up_count = sum(1 for r in current_records if r[3] in ['MOVE_UP', 'NEW_ENTRY'])
    down_count = sum(1 for r in current_records if r[3] == 'MOVE_DOWN')

    print("\n" + "="*50)
    print(f"CURRENT TOP200 CHART DATA ({DATA_CUTOFF_DATE}) FOR {artist_name.upper()}")
    print("="*50)
    
    if not current_records:
        print("Artist is not currently on any Top200 charts.")
    else:
        for r in current_records: 
            print(f"Region: {r[1]:<22} | Song: {r[5][:15]:<15} | Rank: {str(r[2]):<3} | Trend: {r[3]}")
            
        print("-" * 50)
        print(f"UPWARD TREND COUNT: {up_count} | DOWNWARD TREND COUNT: {down_count}")
    print("="*50 + "\n")

In [14]:
print_curr_chart_data('Drake')


CURRENT TOP200 CHART DATA (2021-12-31) FOR DRAKE
Region: Bulgaria               | Song: Fair Trade (wit | Rank: 166 | Trend: MOVE_DOWN
Region: Bulgaria               | Song: Knife Talk (wit | Rank: 111 | Trend: MOVE_DOWN
Region: Bulgaria               | Song: Way 2 Sexy (wit | Rank: 57  | Trend: MOVE_DOWN
Region: United Arab Emirates   | Song: Fair Trade (wit | Rank: 56  | Trend: MOVE_DOWN
Region: United Arab Emirates   | Song: Girls Want Girl | Rank: 173 | Trend: MOVE_UP
Region: United Arab Emirates   | Song: God's Plan      | Rank: 142 | Trend: NEW_ENTRY
Region: United Arab Emirates   | Song: Knife Talk (wit | Rank: 130 | Trend: MOVE_DOWN
Region: United Arab Emirates   | Song: One Dance       | Rank: 65  | Trend: MOVE_UP
Region: United Arab Emirates   | Song: Passionfruit    | Rank: 101 | Trend: NEW_ENTRY
Region: United Arab Emirates   | Song: Way 2 Sexy (wit | Rank: 68  | Trend: MOVE_DOWN
Region: United States          | Song: Fair Trade (wit | Rank: 64  | Trend: MOVE_DOWN
Region: 

In [15]:
print_curr_chart_data('Fall Out Boy')


CURRENT TOP200 CHART DATA (2021-12-31) FOR FALL OUT BOY
Artist is not currently on any Top200 charts.

